In [1]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import astropy

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure, show, savefig, close
import matplotlib.font_manager as fm
import itertools
from matplotlib import rcParams
import scienceplots
import pickle

from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
from matplotlib.patches import Patch

import os
import xarray as xr
from taurex.cache import OpacityCache, CIACache
OpacityCache().set_opacity_path('./xsec/')
CIACache().set_cia_path('./cia/')

from taurex.temperature import TemperatureFile
from taurex.chemistry import TaurexChemistry
from taurex.chemistry import ConstantGas

from taurex.planet import Planet
from taurex.stellar import BlackbodyStar, PhoenixStar

from taurex.model import EmissionModel, TransmissionModel
from taurex.pressure import SimplePressureProfile

from explor.model import HotSpotPhaseCurveModel

from taurex.contributions import AbsorptionContribution
from taurex.contributions import CIAContribution
from taurex.contributions import RayleighContribution

from matplotlib.lines import Line2D

from taurex.binning import FluxBinner
from binning_funcs import *
from scipy.optimize import curve_fit, brentq

In [6]:

# ── Build supertable ──────────────────────────────────────────────────────────
exec(open("supertable_code.py").read())   # defines `df`
# remove LHS1478 rows
df = df[~df["Planet"].str.contains("LHS 1478b")].reset_index(drop=True)

# ── Interactive scatter explorer ──────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import numpy as np

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# ── Widgets ───────────────────────────────────────────────────────────────────
style   = {"description_width": "90px"}
layout  = widgets.Layout(width="320px")

w_x     = widgets.Dropdown(options=numeric_cols, value="M_p [M_earth]",       description="X axis:",    style=style, layout=layout)
w_y     = widgets.Dropdown(options=numeric_cols, value="R_p [R_earth]",       description="Y axis:",    style=style, layout=layout)
w_color = widgets.Dropdown(options=["Retrieval Evidence", "N_obs"],           description="Color by:",  style=style, layout=layout)
w_size  = widgets.Dropdown(options=["uniform"] + numeric_cols,                description="Size by:",   style=style, layout=layout)
w_logx  = widgets.Checkbox(value=False, description="Log X",  style=style)
w_logy  = widgets.Checkbox(value=False, description="Log Y",  style=style)
w_annot = widgets.Checkbox(value=True,  description="Labels", style=style)

out = widgets.Output()

# ── Plot function ─────────────────────────────────────────────────────────────
def update(_=None):
    xcol  = w_x.value
    ycol  = w_y.value
    ccol  = w_color.value
    scol  = w_size.value

    mask     = df[xcol].notna() & df[ycol].notna() & df[ccol].notna()
    plot_df  = df[mask].copy()

    with out:
        out.clear_output(wait=True)

        if len(plot_df) == 0:
            print("No rows with valid values for all selected columns.")
            return

        fig, ax = plt.subplots(figsize=(9, 6))

        # ── Colour ────────────────────────────────────────────────────────────
        cvals = plot_df[ccol].values.astype(float)
        vmin, vmax = np.nanmin(cvals), np.nanmax(cvals)
        norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)
        cmap  = cm.plasma if ccol == "Retrieval Evidence" else cm.viridis

        # ── Size ──────────────────────────────────────────────────────────────
        if scol == "uniform":
            sizes = np.full(len(plot_df), 100.0)
        else:
            svals = plot_df[scol].fillna(plot_df[scol].median()).values.astype(float)
            smin, smax = np.nanmin(svals), np.nanmax(svals)
            sizes = 30 + 250 * (svals - smin) / (smax - smin + 1e-12)

        sc = ax.scatter(
            plot_df[xcol], plot_df[ycol],
            c=cvals, cmap=cmap, norm=norm,
            s=sizes, edgecolors="k", linewidths=0.5, alpha=0.88,
        )

        # ── Labels ────────────────────────────────────────────────────────────
        if w_annot.value:
            for _, row in plot_df.iterrows():
                label = row["Planet"].split()[0] + f"\n#{int(row['Case #'])}"
                ax.annotate(label, (row[xcol], row[ycol]),
                            fontsize=7, ha="center", va="bottom",
                            xytext=(0, 5), textcoords="offset points")

        # ── Colorbar ──────────────────────────────────────────────────────────
        cb = plt.colorbar(sc, ax=ax)
        cb.set_label(ccol, fontsize=11)

        # ── Axes ──────────────────────────────────────────────────────────────
        if w_logx.value:
            ax.set_xscale("log")
        if w_logy.value:
            ax.set_yscale("log")

        size_label = "uniform" if scol == "uniform" else scol
        ax.set_xlabel(xcol, fontsize=12)
        ax.set_ylabel(ycol, fontsize=12)
        ax.set_title(f"{ycol}  vs  {xcol}\ncolor → {ccol}   |   size → {size_label}", fontsize=11)
        ax.grid(True, alpha=0.25, linestyle="--")
        plt.tight_layout()
        plt.show()

# ── Wire callbacks ────────────────────────────────────────────────────────────
for w in [w_x, w_y, w_color, w_size, w_logx, w_logy, w_annot]:
    w.observe(update, names="value")

# ── Layout & display ──────────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HBox([w_x,    w_y]),
    widgets.HBox([w_color, w_size]),
    widgets.HBox([w_logx, w_logy, w_annot]),
])
display(controls, out)
update()


Output()

In [7]:

# ── 2-D interpolated colormap explorer ───────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import numpy as np
from scipy.interpolate import griddata

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# ── Widgets ───────────────────────────────────────────────────────────────────
style  = {"description_width": "110px"}
layout = widgets.Layout(width="340px")

w_x      = widgets.Dropdown(options=numeric_cols, value="M_p [M_earth]",  description="X axis:",       style=style, layout=layout)
w_y      = widgets.Dropdown(options=numeric_cols, value="R_p [R_earth]",  description="Y axis:",       style=style, layout=layout)
w_color  = widgets.Dropdown(options=["Retrieval Evidence", "N_obs"],      description="Color by:",     style=style, layout=layout)
w_method = widgets.Dropdown(options=["linear", "cubic", "nearest"],       description="Interpolation:", style=style, layout=layout)
w_logx   = widgets.Checkbox(value=False, description="Log X",  style=style)
w_logy   = widgets.Checkbox(value=False, description="Log Y",  style=style)
w_pts    = widgets.Checkbox(value=True,  description="Show points", style=style)
w_res    = widgets.IntSlider(value=200, min=50, max=500, step=50,
                             description="Grid res:", style=style,
                             layout=widgets.Layout(width="340px"))

out2 = widgets.Output()

# ── Plot function ─────────────────────────────────────────────────────────────
def update2(_=None):
    xcol   = w_x.value
    ycol   = w_y.value
    ccol   = w_color.value
    method = w_method.value
    res    = w_res.value

    mask    = df[xcol].notna() & df[ycol].notna() & df[ccol].notna()
    plot_df = df[mask].copy()

    with out2:
        out2.clear_output(wait=True)

        if len(plot_df) < 3:
            print("Need at least 3 valid rows to interpolate — try a different color column.")
            return

        xv = plot_df[xcol].values.astype(float)
        yv = plot_df[ycol].values.astype(float)
        cv = plot_df[ccol].values.astype(float)

        # ── Build grid (log-space if requested) ───────────────────────────────
        if w_logx.value and np.all(xv > 0):
            xgrid = np.logspace(np.log10(xv.min()), np.log10(xv.max()), res)
        else:
            xgrid = np.linspace(xv.min(), xv.max(), res)

        if w_logy.value and np.all(yv > 0):
            ygrid = np.logspace(np.log10(yv.min()), np.log10(yv.max()), res)
        else:
            ygrid = np.linspace(yv.min(), yv.max(), res)

        XX, YY = np.meshgrid(xgrid, ygrid)

        # Interpolate in log-space coordinates for the grid, but pass original points
        xi = np.log10(xv) if (w_logx.value and np.all(xv > 0)) else xv
        yi = np.log10(yv) if (w_logy.value and np.all(yv > 0)) else yv
        xi_grid = np.log10(XX) if (w_logx.value and np.all(xv > 0)) else XX
        yi_grid = np.log10(YY) if (w_logy.value and np.all(yv > 0)) else YY

        ZZ = griddata(np.column_stack([xi, yi]), cv,
                      np.column_stack([xi_grid.ravel(), yi_grid.ravel()]),
                      method=method).reshape(res, res)

        # ── Plot ──────────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(9, 6))

        cmap  = cm.plasma if ccol == "Retrieval Evidence" else cm.viridis
        vmin, vmax = np.nanmin(cv), np.nanmax(cv)
        norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)

        im = ax.pcolormesh(XX, YY, ZZ, cmap=cmap, norm=norm,
                           shading="gouraud", alpha=0.85)

        # ── Scatter actual data points on top ─────────────────────────────────
        if w_pts.value:
            ax.scatter(xv, yv, c=cv, cmap=cmap, norm=norm,
                       s=70, edgecolors="white", linewidths=0.8, zorder=5)
            for _, row in plot_df.iterrows():
                label = row["Planet"].split()[0] + f"\n#{int(row['Case #'])}"
                ax.annotate(label, (row[xcol], row[ycol]),
                            fontsize=7, ha="center", va="bottom",
                            xytext=(0, 5), textcoords="offset points",
                            color="white",
                            bbox=dict(boxstyle="round,pad=0.1", fc="black",
                                      alpha=0.35, lw=0))

        cb = plt.colorbar(im, ax=ax)
        cb.set_label(ccol, fontsize=11)

        if w_logx.value and np.all(xv > 0):
            ax.set_xscale("log")
        if w_logy.value and np.all(yv > 0):
            ax.set_yscale("log")

        ax.set_xlabel(xcol, fontsize=12)
        ax.set_ylabel(ycol, fontsize=12)
        ax.set_title(f"{ccol}  —  interpolated over  {xcol}  ×  {ycol}"
                     f"  [{method}]", fontsize=11)
        ax.grid(True, alpha=0.15, linestyle="--", color="white")
        plt.tight_layout()
        plt.show()

# ── Wire callbacks ────────────────────────────────────────────────────────────
for w in [w_x, w_y, w_color, w_method, w_logx, w_logy, w_pts, w_res]:
    w.observe(update2, names="value")

# ── Layout & display ──────────────────────────────────────────────────────────
controls2 = widgets.VBox([
    widgets.HBox([w_x, w_y]),
    widgets.HBox([w_color, w_method]),
    widgets.HBox([w_logx, w_logy, w_pts]),
    w_res,
])
display(controls2, out2)
update2()


Output()